In [ ]:
from anthropic import Anthropic
from anthropic.types import MessageParam
from dotenv import load_dotenv

load_dotenv()

client: Anthropic = Anthropic()
model: str = "claude-sonnet-4-6"

Messages = list[MessageParam]

def add_user_message(messages: Messages, text: str) -> None:
    messages.append({ "role": "user", "content": text })

def add_assistant_message(messages: Messages, text: str) -> None:
    messages.append({ "role": "assistant", "content": text })

def chat(
        messages: Messages,
        system: str|None = None,
        temperature: float = 1.0) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    response = client.messages.create(
        **params
    )
    
    assistant_text = response.content[0].text
    add_assistant_message(messages, assistant_text)
    return assistant_text


In [ ]:
"""
The old way is to prefill assistant message,
but that is not supported with sonnet 4.6+ anymore.

The error message:
"This model does not support assistant message prefill. 
The conversation must end with a user message."

messages: Messages = []

add_user_message(messages, "Generate a very short event bridge rule as JSON.")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
"""


In [ ]:
"""
Use system prompt to instruct model to respond with raw JSON, 
and then parse it with json.loads.
"""

import json

messages: Messages = []
add_user_message(messages, "Generate a very short event bridge rule as JSON.")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    system="Respond only with raw valid JSON. No explanation, no markdown, no code blocks.",
    messages=messages
)

text = response.content[0].text
json.loads(text)

In [ ]:
"""
if we know the schema of the JSON we want, 
we can also provide that to the model and it will try to follow it. 
This is more efficient than asking the model to generate JSON 
and then parsing it ourselves, because the model can generate
the JSON in a way that is easier for us to parse.
"""

import json

messages: Messages = []
add_user_message(messages, "Generate a very short event bridge rule as JSON.")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "Name": {"type": "string"},
                    "EventPattern": {
                        "type": "object",
                        "properties": {
                            "source": {"type": "array", "items": {"type": "string"}},
                            "detail-type": {"type": "array", "items": {"type": "string"}}
                        },
                        "required": ["source", "detail-type"],
                        "additionalProperties": False
                    },
                    "State": {"type": "string", "enum": ["ENABLED", "DISABLED"]},
                    "Description": {"type": "string"}
                },
                "required": ["Name", "EventPattern", "State"],
                "additionalProperties": False
            }
        }
    }
)

text = response.content[0].text
json.loads(text)